In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path

ZIP_PATH = Path("/content/drive/MyDrive/Colab Notebooks/Deep Learning/Food Detection/food41_yolo.zip")

print("ZIP exists:", ZIP_PATH.exists())

ZIP exists: True


In [5]:
!rm -rf /content/food41_yolo
!unzip -q "/content/drive/MyDrive/Colab Notebooks/Deep Learning/Food Detection/food41_yolo.zip" -d /content/

print("Dataset extracted.")

Dataset extracted.


In [6]:
from pathlib import Path

DATASET_ROOT = Path("/content/food41_yolo")

print("Dataset:", DATASET_ROOT.exists())
print("YAML:", (DATASET_ROOT / "dataset.yaml").exists())

for split in ["train", "val", "test"]:
    images = list((DATASET_ROOT / "images" / split).glob("*"))
    labels = list((DATASET_ROOT / "labels" / split).glob("*.txt"))

    print(
        f"{split}: "
        f"{len(images)} images | "
        f"{len(labels)} labels"
    )

Dataset: True
YAML: True
train: 4811 images | 4811 labels
val: 1185 images | 1185 labels
test: 1507 images | 1507 labels


In [7]:
!nvidia-smi

Sun Sep  6 12:54:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
!pip install -q -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 7.6 MB/s eta 0:00:00


In [9]:
import ultralytics
import torch

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics: 8.4.142
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [10]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

print("Model loaded successfully.")

Model loaded successfully.


In [11]:
from pathlib import Path
from collections import defaultdict
import hashlib

DATASET = Path("/content/food41_yolo")

hash_locations = defaultdict(list)

for split in ["train", "val", "test"]:
    image_dir = DATASET / "images" / split

    for image_path in image_dir.iterdir():

        # Skip ECUSTFD images
        if image_path.name.startswith("ecust_"):
            continue

        if image_path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
            continue

        with open(image_path, "rb") as f:
            file_hash = hashlib.sha256(f.read()).hexdigest()

        hash_locations[file_hash].append(
            (split, image_path.name)
        )


# Same exact image occurring more than once
duplicates = {
    h: locations
    for h, locations in hash_locations.items()
    if len(locations) > 1
}

# More important: duplicate image appearing across different splits
cross_split_duplicates = {}

for h, locations in duplicates.items():

    splits = {split for split, _ in locations}

    if len(splits) > 1:
        cross_split_duplicates[h] = locations


print("Unique UEC image hashes:", len(hash_locations))
print("Duplicate UEC image groups:", len(duplicates))
print(
    "Duplicate groups across train/val/test:",
    len(cross_split_duplicates)
)

if cross_split_duplicates:

    print("\nExamples:")

    for i, locations in enumerate(
        cross_split_duplicates.values()
    ):
        print(locations)

        if i >= 9:
            break

else:
    print("\n✓ NO EXACT UEC CROSS-SPLIT DUPLICATES")

Unique UEC image hashes: 4774
Duplicate UEC image groups: 601
Duplicate groups across train/val/test: 345

Examples:
[('train', '12_7160.jpg'), ('val', '68_7160.jpg')]
[('train', '36_13819.jpg'), ('train', '1_13819.jpg'), ('val', '1_14605.jpg'), ('val', '87_14605.jpg'), ('test', '36_14605.jpg')]
[('train', '1_15719.jpg'), ('test', '36_15719.jpg')]
[('train', '36_30.jpg'), ('val', '1_30.jpg')]
[('train', '12_6963.jpg'), ('train', '38_6963.jpg'), ('test', '68_6963.jpg')]
[('train', '17_14054.jpg'), ('test', '87_14054.jpg')]
[('train', '36_13997.jpg'), ('val', '9_13997.jpg')]
[('train', '87_5115.jpg'), ('val', '1_5115.jpg')]
[('train', '1_14910.jpg'), ('test', '98_14910.jpg')]
[('train', '36_11161.jpg'), ('train', '1_11161.jpg'), ('train', '55_11156.jpg'), ('train', '1_11156.jpg'), ('train', '98_11156.jpg'), ('val', '36_11156.jpg'), ('test', '98_11161.jpg')]
